In [3]:
import pandas as pd
import numpy as np
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
df = pd.read_csv("jobs_dataset_1000.csv")
df.head()

,job_title,skills_required,description
0,Data Engineer,Python SQL ETL Big Data Hadoop,Data Engineer role requiring skills in Python ...
1,UI UX Designer,Figma Adobe XD Prototyping Design,UI UX Designer role requiring skills in Figma ...
2,Frontend Developer,HTML CSS JavaScript React,Frontend Developer role requiring skills in HT...
3,QA Engineer,Testing Automation Selenium Debugging,QA Engineer role requiring skills in Testing A...
4,Business Analyst,Excel SQL Communication Analytics,Business Analyst role requiring skills in Exce...


In [5]:
df["job_title"] = df["job_title"].fillna("").astype(str)
df["skills_required"] = df["skills_required"].fillna("").astype(str)
df["description"] = df["description"].fillna("").astype(str)

df = df.drop_duplicates(subset=["job_title", "skills_required", "description"]).reset_index(drop=True)

df.head()

,job_title,skills_required,description
0,Data Engineer,Python SQL ETL Big Data Hadoop,Data Engineer role requiring skills in Python ...
1,UI UX Designer,Figma Adobe XD Prototyping Design,UI UX Designer role requiring skills in Figma ...
2,Frontend Developer,HTML CSS JavaScript React,Frontend Developer role requiring skills in HT...
3,QA Engineer,Testing Automation Selenium Debugging,QA Engineer role requiring skills in Testing A...
4,Business Analyst,Excel SQL Communication Analytics,Business Analyst role requiring skills in Exce...


In [6]:
df["text"] = df["skills_required"] + " " + df["description"]

df[["job_title", "text"]].head()

,job_title,text
0,Data Engineer,Python SQL ETL Big Data Hadoop Data Engineer r...
1,UI UX Designer,Figma Adobe XD Prototyping Design UI UX Design...
2,Frontend Developer,HTML CSS JavaScript React Frontend Developer r...
3,QA Engineer,Testing Automation Selenium Debugging QA Engin...
4,Business Analyst,Excel SQL Communication Analytics Business Ana...


In [7]:
PHRASES = [
    "machine learning",
    "deep learning",
    "data science",
    "data analysis",
    "data structures",
    "natural language processing",
    "computer vision",
    "big data",
    "data visualization",
    "power bi"
]

In [8]:
def extract_skills_as_phrases(text):
    text = str(text).lower()
    found = set()

    # capture multi-word phrases
    for phrase in PHRASES:
        if phrase in text:
            found.add(phrase)
            text = text.replace(phrase, " ")

    # capture single words
    for token in text.replace(",", " ").split():
        token = token.strip()
        if token:
            found.add(token)

    return found

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["text"])

print("TF-IDF trained")
print(tfidf_matrix.shape)

TF-IDF trained
(20, 92)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

user_input = "python, machine learning, deep learning, sql"

user_vector = tfidf.transform([user_input])
similarity = cosine_similarity(user_vector, tfidf_matrix)[0]

sorted_indices = np.argsort(similarity)[::-1][:100]

recommended_jobs = df.iloc[sorted_indices].copy()
recommended_jobs["match_score"] = similarity[sorted_indices] * 100

recommended_jobs = recommended_jobs.drop_duplicates(subset=["job_title"])
recommended_jobs = recommended_jobs.head(5)

recommended_jobs[["job_title", "match_score"]]

,job_title,match_score
7,AI Engineer,83.406008
14,ML Engineer,52.872871
10,Data Scientist,52.515131
12,Full Stack Developer,17.193405
11,Data Analyst,16.196760


In [11]:
user_skills = set(s.strip().lower() for s in user_input.split(",") if s.strip())

def skill_gap(row):
    job_skills = extract_skills_as_phrases(row["skills_required"])
    matched = sorted(list(user_skills & job_skills))
    missing = sorted(list(job_skills - user_skills))
    return matched, missing

recommended_jobs["matched_skills"], recommended_jobs["missing_skills"] = zip(
    *recommended_jobs.apply(skill_gap, axis=1)
)

recommended_jobs[["job_title", "matched_skills", "missing_skills"]]

,job_title,matched_skills,missing_skills
7,AI Engineer,"[deep learning, machine learning, python]",[nlp]
14,ML Engineer,"[deep learning, python]","[pytorch, tensorflow]"
10,Data Scientist,"[machine learning, python]","[pandas, statistics]"
12,Full Stack Developer,"[python, sql]","[css, html, javascript]"
11,Data Analyst,"[python, sql]","[data visualization, excel]"
